In [ ]:
#HELPERS

# print(f"Current Position: {reachy.mobile_base.odometry}")

# This will return a list of obstacles or a safety state depending on your exact SDK version
#print(f"Safety state: {mobile_base.safety_state}")

# Ensure we are in a mode that allows navigation
#print(f"Current Drive Mode: {mobile_base.drive_mode}")

#SET SPEED
# reachy.mobile_base.set_speed(x_vel=0.5, y_vel=0.0, rot_vel=0)

In [3]:
import math,sys,time,traceback, pygame, os
from signal import SIGINT, signal

#from reachy_sdk import ReachySDK
from mobile_base_sdk import MobileBaseSDK
from reachy_sdk import ReachySDK

#mobile_base = MobileBaseSDK(host='localhost')

# Connect to both Reachy and the Mobile Base
reachy = ReachySDK(host='localhost', with_mobile_base=True)
#reachy = ReachySDK(host='localhost')

# The mobile base is now accessible via:
# reachy.mobile_base

pygame 2.4.0 (SDL 2.26.4, Python 3.10.6)
Hello from the pygame community. https://www.pygame.org/contribute.html


OSError: Failed to connect to mobile base!

In [ ]:
#Battery voltage check
print(f"Battery: {reachy.mobile_base.battery_voltage}V")

In [16]:
#RESET ODOMETRY
reachy.mobile_base.reset_odometry()
#mobile_base.goto(x=0, y=0, theta=90)

In [19]:
reachy.mobile_base.reset_odometry()
reachy.mobile_base.goto(0, 0, -20)

In [20]:
"""
follow_person.py
----------------
Makes Reachy follow a person using the mobile base and the SDK camera.

Camera access:
    Uses reachy.left_camera.last_frame (or right_camera) via the Reachy SDK,
    matching the pattern from SDKCameraViewer. No cv2.VideoCapture needed.

Logic:
  - Detect a person via OpenCV's HOG pedestrian detector.
  - If the person's bounding box is SMALLER than TARGET_BOX_FRACTION → move forward.
  - If the person's bounding box is LARGER than TARGET_BOX_FRACTION  → move backward.
  - If the person's horizontal center deviates from frame center      → rotate.

Dependencies:
    pip install opencv-python numpy reachy-sdk

Usage (Jupyter / script):
    from follow_person import follow_person
    follow_person(reachy, camera_side='left', show_preview=True)
"""

import time
import cv2
import numpy as np

# ── Tuneable constants ────────────────────────────────────────────────────────

# Which camera to use: 'left' or 'right'
CAMERA_SIDE = 'left'

# Target bounding-box area as a fraction of total frame area (0–1).
# Increase to follow closer; decrease to keep more distance.
TARGET_BOX_FRACTION = 0.12        # ~12 % of frame = "right distance"
AREA_DEAD_ZONE      = 0.02        # ± tolerance before moving fwd/bwd

# Horizontal centering – fraction of frame width allowed off-centre
CENTER_DEAD_ZONE_FRACTION = 0.10  # ±10 % of frame width = "centred enough"

# Motion step sizes issued per control tick
FWD_STEP  =  0.3    # metres  – move forward
BWD_STEP  = -0.2    # metres  – back off
ROT_STEP  =  0.15   # radians – rotate (CCW positive, matching goto convention)

# Control loop rate
LOOP_HZ = 5

# ── HOG person detector ───────────────────────────────────────────────────────

hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())


def detect_largest_person(frame):
    """Return (x, y, w, h) of the largest detected person, or None."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    boxes, _ = hog.detectMultiScale(
        gray,
        winStride=(8, 8),
        padding=(4, 4),
        scale=1.05,
    )
    if len(boxes) == 0:
        return None
    areas = [w * h for (x, y, w, h) in boxes]
    return boxes[np.argmax(areas)]


# ── Control logic ─────────────────────────────────────────────────────────────

def compute_goto(box, frame_w, frame_h):
    """
    Compute (x, y, theta) for reachy.mobile_base.goto() given a bounding box.

    x     – forward/backward displacement (metres)
    y     – always 0 (no lateral movement)
    theta – rotation (radians, CCW positive)
    """
    x_box, _, w_box, h_box = box
    box_fraction  = (w_box * h_box) / (frame_w * frame_h)
    area_error    = box_fraction - TARGET_BOX_FRACTION

    # Forward / backward
    if abs(area_error) < AREA_DEAD_ZONE:
        move_x = 0.0
    elif area_error < 0:   # too far → move forward
        move_x = FWD_STEP
    else:                  # too close → back off
        move_x = BWD_STEP

    # Rotation to centre the person horizontally
    box_center_x          = x_box + w_box / 2
    horiz_error_fraction  = (box_center_x - frame_w / 2) / frame_w

    if abs(horiz_error_fraction) < CENTER_DEAD_ZONE_FRACTION:
        rotate_theta = 0.0
    else:
        # Person right of centre → rotate CW (negative theta)
        # Person left of centre  → rotate CCW (positive theta)
        rotate_theta = -np.sign(horiz_error_fraction) * ROT_STEP

    return move_x, 0.0, rotate_theta


# ── Main follow loop ──────────────────────────────────────────────────────────

def follow_person(reachy, camera_side=CAMERA_SIDE, show_preview=False):
    """
    Main follow loop.

    Args:
        reachy:       A connected ReachySDK instance.
        camera_side:  'left' or 'right' – which of Reachy's cameras to use.
        show_preview: Show a live OpenCV window with the detection overlay.
    """
    # Access the camera the same way SDKCameraViewer does
    camera = getattr(reachy, f'{camera_side}_camera')

    # Warm up – wait until we get a real frame
    print(f"Waiting for first frame from {camera_side}_camera…")
    frame = None
    for _ in range(50):          # up to ~5 s
        frame = camera.last_frame
        if frame is not None:
            break
        time.sleep(0.1)

    if frame is None:
        raise RuntimeError(
            f"No frame received from {camera_side}_camera after 5 s. "
            "Is reachy_sdk_server running?"
        )

    print(f"Camera ready ({frame.shape[1]}×{frame.shape[0]}). Following…")
    print("Press Ctrl+C to stop.")

    tick = 1.0 / LOOP_HZ

    try:
        while True:
            loop_start = time.monotonic()

            frame = camera.last_frame
            if frame is None:
                print("Warning: empty frame, skipping tick…")
                time.sleep(tick)
                continue

            frame_h, frame_w = frame.shape[:2]
            box = detect_largest_person(frame)

            if box is None:
                print("No person detected – holding position.")
            else:
                x, y, theta = compute_goto(box, frame_w, frame_h)
                area_pct = box[2] * box[3] / (frame_w * frame_h) * 100

                print(
                    f"box={box}  area={area_pct:.1f}%  "
                    f"→ goto(x={x:.2f}, y={y:.2f}, θ={theta:.2f} rad)"
                )

                if x != 0.0 or theta != 0.0:
                    reachy.mobile_base.goto(x, y, theta)

            # Optional live preview
            if show_preview:
                vis = frame.copy()
                if box is not None:
                    bx, by, bw, bh = box
                    cv2.rectangle(vis, (bx, by), (bx + bw, by + bh), (0, 255, 0), 2)
                # Draw frame centre line
                cv2.line(vis, (frame_w // 2, 0), (frame_w // 2, frame_h), (255, 80, 0), 1)
                cv2.imshow("Reachy Follow", vis)
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break

            # Pace the loop
            elapsed = time.monotonic() - loop_start
            remainder = tick - elapsed
            if remainder > 0:
                time.sleep(remainder)

    except KeyboardInterrupt:
        print("\nStopped by user.")
    finally:
        if show_preview:
            cv2.destroyAllWindows()
        reachy.mobile_base.goto(0, 0, 0)   # stop the robot
        print("Mobile base stopped.")


# ── Entry point ───────────────────────────────────────────────────────────────

# if __name__ == "__main__":
#     from reachy_sdk import ReachySDK

#     reachy = ReachySDK('localhost')   # replace with robot IP if remote
#     follow_person(reachy, camera_side='left', show_preview=True)

In [ ]:
follow_person(reachy, camera_side='left', show_preview=True)

Waiting for first frame from left_camera…
Camera ready (480×640). Following…
Press Ctrl+C to stop.
box=[136 192 198 397]  area=25.6%  → goto(x=-0.20, y=0.00, θ=0.00 rad)


In [1]:
"""
follow_person.py  (revised)
----------------------------
Changes vs original:
  1. Loop runs in a daemon Thread; call stop_follow() from any notebook cell.
  2. Live preview is served as an MJPEG stream on http://<reachy-ip>:5801
     so you can watch it in any browser on your laptop — no X11 forwarding needed.
  3. stop_follow() also sends a final goto(0,0,0) to halt the base.
"""

import time
import threading
import cv2
import numpy as np
from http.server import BaseHTTPRequestHandler, HTTPServer

# ── Tuneable constants ────────────────────────────────────────────────────────
CAMERA_SIDE               = 'left'
TARGET_BOX_FRACTION       = 0.12
AREA_DEAD_ZONE            = 0.02
CENTER_DEAD_ZONE_FRACTION = 0.10
FWD_STEP                  =  0.3
BWD_STEP                  = -0.2
ROT_STEP                  =  0.15
LOOP_HZ                   = 5
MJPEG_PORT                = 5801      # open http://<robot-ip>:5801 on laptop

# ── Shared state ──────────────────────────────────────────────────────────────
_stop_event   = threading.Event()
_latest_jpeg  = None                  # bytes – updated by control loop
_jpeg_lock    = threading.Lock()
_follow_thread = None

# ── HOG detector ─────────────────────────────────────────────────────────────
hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

def detect_largest_person(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    boxes, _ = hog.detectMultiScale(gray, winStride=(8,8), padding=(4,4), scale=1.05)
    if len(boxes) == 0:
        return None
    return boxes[np.argmax([w*h for (x,y,w,h) in boxes])]

def compute_goto(box, frame_w, frame_h):
    x_box, _, w_box, h_box = box
    area_error = (w_box * h_box) / (frame_w * frame_h) - TARGET_BOX_FRACTION
    move_x = (0.0 if abs(area_error) < AREA_DEAD_ZONE
              else FWD_STEP if area_error < 0 else BWD_STEP)
    horiz_err = (x_box + w_box/2 - frame_w/2) / frame_w
    rotate_theta = (0.0 if abs(horiz_err) < CENTER_DEAD_ZONE_FRACTION
                    else -np.sign(horiz_err) * ROT_STEP)
    return move_x, 0.0, rotate_theta

# ── MJPEG server ──────────────────────────────────────────────────────────────
class _MjpegHandler(BaseHTTPRequestHandler):
    def log_message(self, *args):   # silence access log spam
        pass

    def do_GET(self):
        self.send_response(200)
        self.send_header('Content-Type',
                         'multipart/x-mixed-replace; boundary=--jpgboundary')
        self.end_headers()
        try:
            while not _stop_event.is_set():
                with _jpeg_lock:
                    jpg = _latest_jpeg
                if jpg is None:
                    time.sleep(0.05)
                    continue
                self.wfile.write(b'--jpgboundary\r\n'
                                 b'Content-Type: image/jpeg\r\n\r\n')
                self.wfile.write(jpg)
                self.wfile.write(b'\r\n')
                time.sleep(1 / LOOP_HZ)
        except (BrokenPipeError, ConnectionResetError):
            pass   # browser tab closed – that's fine

def _run_mjpeg_server():
    srv = HTTPServer(('0.0.0.0', MJPEG_PORT), _MjpegHandler)
    srv.timeout = 1
    while not _stop_event.is_set():
        srv.handle_request()

# ── Control loop (runs in background thread) ──────────────────────────────────
def _control_loop(reachy, camera_side):
    global _latest_jpeg
    camera = getattr(reachy, f'{camera_side}_camera')

    print(f"Waiting for first frame from {camera_side}_camera…")
    frame = None
    for _ in range(50):
        frame = camera.last_frame
        if frame is not None:
            break
        time.sleep(0.1)
    if frame is None:
        print("ERROR: no frame received – aborting.")
        return

    print(f"Camera ready ({frame.shape[1]}×{frame.shape[0]}). Following…")
    print(f"Preview → http://<reachy-ip>:{MJPEG_PORT}  (open in laptop browser)")

    tick = 1.0 / LOOP_HZ
    while not _stop_event.is_set():
        t0 = time.monotonic()
        frame = camera.last_frame
        if frame is None:
            time.sleep(tick)
            continue

        fh, fw = frame.shape[:2]
        box = detect_largest_person(frame)
        vis = frame.copy()

        if box is None:
            print("No person detected – holding.")
        else:
            x, y, theta = compute_goto(box, fw, fh)
            bx, by, bw, bh = box
            print(f"box={box}  area={bw*bh/(fw*fh)*100:.1f}%  "
                  f"→ goto(x={x:.2f} y={y:.2f} θ={theta:.2f})")
            if x != 0.0 or theta != 0.0:
                reachy.mobile_base.goto(x, y, theta)
            cv2.rectangle(vis, (bx, by), (bx+bw, by+bh), (0,255,0), 2)

        cv2.line(vis, (fw//2, 0), (fw//2, fh), (255,80,0), 1)

        # Encode frame → JPEG for MJPEG stream
        ok, jpg_buf = cv2.imencode('.jpg', vis, [cv2.IMWRITE_JPEG_QUALITY, 70])
        if ok:
            with _jpeg_lock:
                _latest_jpeg = jpg_buf.tobytes()

        elapsed = time.monotonic() - t0
        remainder = tick - elapsed
        if remainder > 0:
            time.sleep(remainder)

    # ── cleanup on exit ──
    try:
        reachy.mobile_base.goto(0, 0, 0)
        print("Mobile base stopped.")
    except Exception as e:
        print(f"Stop command failed: {e}")

# ── Public API ────────────────────────────────────────────────────────────────
def follow_person(reachy, camera_side=CAMERA_SIDE):
    """Start following in background threads. Returns immediately."""
    global _follow_thread
    _stop_event.clear()

    mjpeg_thread = threading.Thread(target=_run_mjpeg_server, daemon=True)
    mjpeg_thread.start()

    _follow_thread = threading.Thread(
        target=_control_loop, args=(reachy, camera_side), daemon=True)
    _follow_thread.start()
    print("follow_person() started. Call stop_follow() to stop.")

def stop_follow():
    """Stop the follow loop cleanly from any notebook cell."""
    _stop_event.set()
    if _follow_thread is not None:
        _follow_thread.join(timeout=3)
    print("Follow loop stopped.")

In [2]:
# from follow_person import follow_person
follow_person(reachy, camera_side='left')
# → open http://<reachy-ip>:5801 in your laptop browser to see the stream

# Cell 2 – stop (run this whenever you want, even after a crash)
# from follow_person import stop_follow
# 

NameError: name 'reachy' is not defined

In [ ]:
stop_follow()